# Walmart Sales Forecasting — Inference & Submission (2nd Place)

**მიზანი:** მე-2 ადგილის მოდელით (**XGBoost v2**) test set-ზე პროგნოზი და Kaggle submission-ის მომზადება.

## 10 მოდელის საბოლოო შედარება

| Rank | Model | Val WMAE | Category | Owner |
|------|-------|----------|----------|-------|
| 1 | TimesFM (zero-shot) | 1309.76 | Foundation | giga |
| 2 | **XGBoost** | **1321.74** | Tree-based | Zaqaria |
| 3 | Prophet (full population) | 1331.57 | Classical | giga |
| 4 | Ensemble (XGBoost + Prophet, weighted blend) | 1332.23 | Ensemble | Team |
| 5 | LightGBM | 1342.90 | Tree-based | giga |
| 6 | N-BEATS | 1378.04 | DL (feed-forward) | Zaqaria |
| 7 | PatchTST | 1420.44 | DL (transformer) | Zaqaria |
| 8 | DLinear | 1494.80 | DL (linear) | giga |
| 9 | TFT | 1538.41 | DL (LSTM + attention) | Zaqaria |
| 10 | SARIMA | 7012.96 | Classical | Zaqaria |

**მე-2 ადგილი:** XGBoost (v2) — TimesFM-ს (1309.76) უკან, 1321.74 WMAE-ით. ეს notebook:
1. Load-ს XGBoost Pipeline-ს Drive-იდან (backup) ან DagsHub Model Registry-იდან
2. აწარმოებს პროგნოზებს **raw** test.csv-ზე (Pipeline-ის შიგნით preprocessing, lag/rolling ისტორია და Dept×Week სეზონური ფიჩერების ჩათვლით)
3. ქმნის Kaggle submission CSV-ს (`Id, Weekly_Sales` format)


## 1. Setup

In [1]:
!pip install mlflow dagshub xgboost --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.

In [2]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

import mlflow
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

print(f"MLflow: {mlflow.__version__}")

MLflow: 3.14.0


In [3]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/walmart'
DATA_DIR = f'{PROJECT_DIR}/data'
MODELS_DIR = f'{PROJECT_DIR}/models'
SUBMISSIONS_DIR = f'{PROJECT_DIR}/submissions'

import os
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

print(f"Data:        {DATA_DIR}")
print(f"Models:      {MODELS_DIR}")
print(f"Submissions: {SUBMISSIONS_DIR}")

Mounted at /content/drive
Data:        /content/drive/MyDrive/walmart/data
Models:      /content/drive/MyDrive/walmart/models
Submissions: /content/drive/MyDrive/walmart/submissions


In [4]:
# DagsHub connection (Registry-სთვის)
import dagshub

DAGSHUB_USERNAME = "zberi23"
DAGSHUB_REPO = "walmart-forecasting"

dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=DAGSHUB_REPO, mlflow=True)

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=a3f25875-f105-441e-b8af-d24144f25a27&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=10f8c4a50f6a2c18505594a222150147df87fb5daa50554cca8ddad573931c76




Accessing as gbera23-dev

Initialized MLflow to track repo "zberi23/walmart-forecasting"

Repository zberi23/walmart-forecasting initialized!

MLflow tracking URI: https://dagshub.com/zberi23/walmart-forecasting.mlflow


## 2. Custom Preprocessor Class

Pipeline pickle-იდან რომ იტვირთოს, კლასის definition-ი Python-ის memory-ში უნდა იყოს. აქ ხელახლა ვაცხადებთ.

In [5]:
class WalmartPreprocessor(BaseEstimator, TransformerMixin):
    """
    v2 preprocessor — ზუსტად იგივე, რაც model_experiment_XGBoostv2.ipynb-ში.
    Pipeline pickle-ის loaded state-ს სჭირდება ეს ზუსტი transform() logic
    (calendar/store ფიჩერები + Store×Dept lag/rolling ისტორია + Dept×Week
    სეზონური საშუალო), თორემ predict() მცდარ ან არასრულ ფიჩერებზე იმუშავებს.
    """

    def __init__(self, stores_df, features_df):
        self.stores_df = stores_df.copy()
        self.features_df = features_df.copy()

        if not pd.api.types.is_datetime64_any_dtype(self.features_df['Date']):
            self.features_df['Date'] = pd.to_datetime(self.features_df['Date'])

        self._dept_rank_map = None

    def fit(self, X, y=None):
        dept_counts = X['Dept'].value_counts()
        self._dept_rank_map = {d: r for r, d in enumerate(dept_counts.index)}

        if y is None and 'Weekly_Sales' in X.columns:
            y = X['Weekly_Sales'].values

        hist = X[['Store', 'Dept', 'Date']].copy()
        if not pd.api.types.is_datetime64_any_dtype(hist['Date']):
            hist['Date'] = pd.to_datetime(hist['Date'])
        hist['Weekly_Sales'] = np.asarray(y, dtype=float)
        hist = hist.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

        grp = hist.groupby(['Store', 'Dept'])['Weekly_Sales']

        hist['lag_1'] = grp.shift(1)
        hist['lag_52'] = grp.shift(52)
        hist['roll_4_mean'] = grp.transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
        hist['roll_12_mean'] = grp.transform(lambda s: s.shift(1).rolling(12, min_periods=1).mean())
        hist['store_dept_expanding_median'] = grp.transform(lambda s: s.shift(1).expanding(min_periods=1).median())

        lag_feature_cols = ['lag_1', 'lag_52', 'roll_4_mean', 'roll_12_mean', 'store_dept_expanding_median']

        global_median = hist['Weekly_Sales'].median()
        self._global_fallback = {col: global_median for col in lag_feature_cols}
        self._lag_feature_cols = lag_feature_cols

        self._history_features = hist[['Store', 'Dept', 'Date'] + lag_feature_cols] \
            .sort_values('Date').reset_index(drop=True)

        dw = hist.copy()
        dw['Week'] = dw['Date'].dt.isocalendar().week.astype(int)
        self._dept_week_table = dw.groupby(['Dept', 'Week'])['Weekly_Sales'].mean()
        self._dept_week_global_mean = dw['Weekly_Sales'].mean()

        return self

    def transform(self, X):
        df = X.copy()

        if not pd.api.types.is_datetime64_any_dtype(df['Date']):
            df['Date'] = pd.to_datetime(df['Date'])

        df = df.merge(self.features_df, on=['Store', 'Date', 'IsHoliday'], how='left')
        df = df.merge(self.stores_df, on='Store', how='left')

        for col in ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']:
            df[col] = df[col].fillna(0)

        for col in ['CPI', 'Unemployment', 'Temperature', 'Fuel_Price']:
            df[col] = df.groupby('Store')[col].transform(
                lambda x: x.fillna(method='ffill').fillna(method='bfill')
            )

        df['Year'] = df['Date'].dt.year
        df['Month'] = df['Date'].dt.month
        df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
        df['Day_of_Year'] = df['Date'].dt.dayofyear
        df['Quarter'] = df['Date'].dt.quarter
        df['WeekOfMonth'] = ((df['Date'].dt.day - 1) // 7 + 1).astype(int)

        super_bowl = pd.to_datetime(['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08'])
        labor_day = pd.to_datetime(['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06'])
        thanksgiving = pd.to_datetime(['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29'])
        christmas = pd.to_datetime(['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27'])
        all_holidays = super_bowl.append(labor_day).append(thanksgiving).append(christmas)

        df['Is_SuperBowl'] = df['Date'].isin(super_bowl).astype(int)
        df['Is_LaborDay'] = df['Date'].isin(labor_day).astype(int)
        df['Is_Thanksgiving'] = df['Date'].isin(thanksgiving).astype(int)
        df['Is_Christmas'] = df['Date'].isin(christmas).astype(int)

        holiday_sorted = pd.Series(sorted(all_holidays))
        def days_to_nearest(d):
            diffs = (holiday_sorted - d).dt.days
            return diffs.abs().min()
        df['Days_To_Holiday'] = df['Date'].apply(days_to_nearest)

        df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
        df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)
        df['Week_Sin'] = np.sin(2 * np.pi * df['Week'] / 52)
        df['Week_Cos'] = np.cos(2 * np.pi * df['Week'] / 52)

        markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
        df['MarkDown_Total'] = df[markdown_cols].sum(axis=1)
        df['MarkDown_Active_Count'] = (df[markdown_cols] > 0).sum(axis=1)
        df['Size_Per_Dept_Rank'] = df['Dept'].map(self._dept_rank_map).fillna(-1)

        size_bins = [0, 50000, 100000, 150000, 200000, np.inf]
        df['Size_Bucket'] = pd.cut(df['Size'], bins=size_bins, labels=False)

        type_mapping = {'A': 0, 'B': 1, 'C': 2}
        df['Type_Encoded'] = df['Type'].map(type_mapping)
        df['IsHoliday'] = df['IsHoliday'].astype(int)

        dw_index = pd.MultiIndex.from_arrays([df['Dept'], df['Week']])
        df['Dept_Week_Seasonal_Avg'] = dw_index.map(self._dept_week_table).to_numpy()
        df['Dept_Week_Seasonal_Avg'] = pd.Series(df['Dept_Week_Seasonal_Avg']).fillna(self._dept_week_global_mean).values

        left = df[['Store', 'Dept', 'Date']].sort_values('Date')
        merged_lags = pd.merge_asof(
            left,
            self._history_features,
            on='Date',
            by=['Store', 'Dept'],
            direction='backward'
        ).set_index(left.index).sort_index()
        for col in self._lag_feature_cols:
            df[col] = merged_lags[col].fillna(self._global_fallback[col])

        df = df.drop(columns=['Date', 'Type'])
        return df


class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, features):
        self.features = features
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[self.features]


## 3. საუკეთესო მოდელის ჩატვირთვა (XGBoost Pipeline)

ორი წყარო:
1. **Drive backup** (joblib.pkl) — სწრაფი, საიმედო
2. **DagsHub Model Registry** — versioned, official

პრიორიტეტი Drive backup-ს, fallback Registry.

In [6]:
# Drive backup-ის ჩატვირთვა
xgb_pipeline_path = f'{MODELS_DIR}/xgboost_pipeline.pkl'

try:
    xgb_pipeline = joblib.load(xgb_pipeline_path)
    print(f"Pipeline ჩაიტვირთა Drive-იდან: {xgb_pipeline_path}")
    print(f"Pipeline steps:")
    for name, step in xgb_pipeline.steps:
        print(f"  {name}: {type(step).__name__}")
except FileNotFoundError:
    print(f"Drive backup ვერ მოიძებნა: {xgb_pipeline_path}")
    print("Fallback: DagsHub Registry-იდან ვცდით ჩატვირთვას")

    # Fallback: Model Registry
    model_uri = "models:/walmart_xgboost/1"
    xgb_pipeline = mlflow.sklearn.load_model(model_uri)
    print(f"Pipeline ჩაიტვირთა Registry-იდან: {model_uri}")

Pipeline ჩაიტვირთა Drive-იდან: /content/drive/MyDrive/walmart/models/xgboost_pipeline.pkl
Pipeline steps:
  preprocess: WalmartPreprocessor
  select: FeatureSelector
  model: XGBRegressor


## 4. Raw test.csv-ის ჩატვირთვა


In [7]:
test_raw = pd.read_csv(f'{DATA_DIR}/test.csv.zip')
test_raw['Date'] = pd.to_datetime(test_raw['Date'])

print(f"Raw test shape: {test_raw.shape}")
print(f"Date range: {test_raw['Date'].min()} → {test_raw['Date'].max()}")
print(f"\nFirst 5 rows:")
test_raw.head()

Raw test shape: (115064, 4)
Date range: 2012-11-02 00:00:00 → 2013-07-26 00:00:00

First 5 rows:


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


## 5. Inference — პროგნოზები raw test-ზე

Pipeline იღებს raw test-ს, შიგნით ატარებს preprocessing-ს (merge, fillna, feature engineering) და აბრუნებს XGBoost-ის პროგნოზებს.

In [8]:
# Pipeline პირდაპირ raw test-ზე
test_predictions = xgb_pipeline.predict(test_raw)

print(f"Raw test rows:     {len(test_raw):,}")
print(f"Predictions count: {len(test_predictions):,}")

print(f"\nProgressive stats:")
print(f"  Min prediction:    {test_predictions.min():.2f}")
print(f"  Max prediction:    {test_predictions.max():.2f}")
print(f"  Mean prediction:   {test_predictions.mean():.2f}")
print(f"  Median prediction: {np.median(test_predictions):.2f}")

# რამდენიმე ცხრილი
print(f"\nFirst 10 predictions:")
sample = test_raw.head(10).copy()
sample['Predicted_Sales'] = test_predictions[:10]
print(sample[['Store', 'Dept', 'Date', 'IsHoliday', 'Predicted_Sales']].to_string(index=False))

Raw test rows:     115,064
Predictions count: 115,064

Progressive stats:
  Min prediction:    -14566.58
  Max prediction:    592015.00
  Mean prediction:   16485.24
  Median prediction: 7784.95

First 10 predictions:
 Store  Dept       Date  IsHoliday  Predicted_Sales
     1     1 2012-11-02      False     39535.980469
     1     1 2012-11-09      False     22503.546875
     1     1 2012-11-16      False     18644.820312
     1     1 2012-11-23       True     20120.441406
     1     1 2012-11-30      False     21239.019531
     1     1 2012-12-07      False     30709.820312
     1     1 2012-12-14      False     40512.656250
     1     1 2012-12-21      False     47535.152344
     1     1 2012-12-28       True     26516.546875
     1     1 2013-01-04      False     17229.308594


## 6. Kaggle Submission Format



In [9]:
# Submission DataFrame-ის შექმნა
submission = pd.DataFrame({
    'Id': (
        test_raw['Store'].astype(str) + '_' +
        test_raw['Dept'].astype(str) + '_' +
        test_raw['Date'].dt.strftime('%Y-%m-%d')
    ),
    'Weekly_Sales': test_predictions
})

# უარყოფითი პროგნოზები 0-ით ვცვლით (გაყიდვები ვერ იქნება ნეგატიური)
n_negative = (submission['Weekly_Sales'] < 0).sum()
if n_negative > 0:
    print(f"Negative predictions clipped: {n_negative}")
    submission['Weekly_Sales'] = submission['Weekly_Sales'].clip(lower=0)

print(f"\nSubmission shape: {submission.shape}")
print(f"\nSubmission head:")
print(submission.head(10).to_string(index=False))

Negative predictions clipped: 4738

Submission shape: (115064, 2)

Submission head:
            Id  Weekly_Sales
1_1_2012-11-02  39535.980469
1_1_2012-11-09  22503.546875
1_1_2012-11-16  18644.820312
1_1_2012-11-23  20120.441406
1_1_2012-11-30  21239.019531
1_1_2012-12-07  30709.820312
1_1_2012-12-14  40512.656250
1_1_2012-12-21  47535.152344
1_1_2012-12-28  26516.546875
1_1_2013-01-04  17229.308594


## 7. Submission CSV-ის შენახვა

In [10]:
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
submission_path = f'{SUBMISSIONS_DIR}/xgboost_2ndplace_submission_{timestamp}.csv'
submission.to_csv(submission_path, index=False)

latest_path = f'{SUBMISSIONS_DIR}/submission_2ndplace_latest.csv'
submission.to_csv(latest_path, index=False)

print(f"Submission შენახულია:")
print(f"  {submission_path}")
print(f"  {latest_path}")
print(f"\nRows: {len(submission):,}")
print(f"File size: {os.path.getsize(submission_path) / 1024:.1f} KB")

Submission შენახულია:
  /content/drive/MyDrive/walmart/submissions/xgboost_2ndplace_submission_20260724_061542.csv
  /content/drive/MyDrive/walmart/submissions/submission_2ndplace_latest.csv

Rows: 115,064
File size: 2927.8 KB


## 8. შეჯამება

### რა გავაკეთეთ

1. **Load** — XGBoost v2 Pipeline (val WMAE 1321.74 — მე-2 ადგილი, TimesFM-ის (1309.76) უკან) Drive backup-იდან
2. **Inference** — raw test.csv-ზე Pipeline-მა ავტომატურად გააკეთა preprocessing (calendar/store ფიჩერები + lag/rolling ისტორია + Dept×Week სეზონური საშუალო) + prediction
3. **Format** — Kaggle-ის სპეციფიკური `Id, Weekly_Sales` submission
4. **Save** — timestamped + `submission_2ndplace_latest.csv` Drive-ზე


### Pipeline-ის სტრუქტურა

Pipeline მუშაობს პირდაპირ raw test set-ზე — preprocessing შიგნით ჩაშენდა:

```
raw test.csv → WalmartPreprocessor → FeatureSelector → XGBoost → predictions
```

ჩაშენებული Custom Transformer-ი ავტომატურად აკეთებს:
- Merge stores + features
- Missing values (MarkDown → 0, CPI/Unemployment/Temperature/Fuel_Price → ffill/bfill)
- Calendar + holiday feature engineering (year, month, week, cyclical, Days_To_Holiday)
- Store×Dept lag/rolling ისტორია (lag_1, lag_52, roll_4_mean, roll_12_mean, expanding median)
- Dept×Week სეზონური საშუალო
- Store type encoding

### Kaggle Submission

| Score Type | WMAE |
|-----------|------|
| Private Score | 2717.26 |
| Public Score | 2647.91 |

### ლინკები

- **MLflow (DagsHub):** https://dagshub.com/zberi23/walmart-forecasting.mlflow
- **WandB (Zaqaria):** https://wandb.ai/zberi23_ml/walmart-forecasting
- **WandB (Giga):** https://wandb.ai/gbera23-free-university-of-tbilisi-/walmart-forecasting